In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
import requests
import pandas as pd
import numpy as np
import lightkurve as lk

In [ ]:
# ============================================================
# 1. Hybrid method functions
# ============================================================

def lafler_kinman_theta(time, mag, periods):
    """
    Lafler-Kinman phase-dispersion statistic.
    Smaller theta = better period.
    """

    theta = np.zeros(len(periods))
    mean_mag = np.nanmean(mag)
    denom = np.nansum((mag - mean_mag)**2)

    for i, P in enumerate(periods):
        phase = (time / P) % 1.0
        order = np.argsort(phase)

        m = mag[order]
        dm = np.diff(np.r_[m, m[0]])

        theta[i] = np.nansum(dm**2) / denom

    return theta


In [ ]:
def hybrid_period_search(
    time,
    mag,
    err=None,
    label="dataset",
    min_period=0.2,
    max_period=100,
    n_periods=5000
):
    """
    Hybrid period search combining Lomb-Scargle and Lafler-Kinman.
    """

    time = np.asarray(time)
    mag = np.asarray(mag)

    good = np.isfinite(time) & np.isfinite(mag)

    if err is not None:
        err = np.asarray(err)
        good &= np.isfinite(err) & (err > 0)

    time = time[good]
    mag = mag[good]

    if err is not None:
        err = err[good]

    # Period grid
    periods = np.logspace(
        np.log10(min_period),
        np.log10(max_period),
        n_periods
    )
    freqs = 1.0 / periods

    # Lomb-Scargle
    if err is not None:
        ls = LombScargle(time, mag, err)
    else:
        ls = LombScargle(time, mag)

    ls_power = ls.power(freqs)
    ls_power = ls_power / np.nanmax(ls_power)

    # Lafler-Kinman
    theta = lafler_kinman_theta(time, mag, periods)

    lk_power = 1.0 / theta
    lk_power = lk_power / np.nanmax(lk_power)

    # Hybrid statistic
    psi = ls_power * lk_power
    psi = psi / np.nanmax(psi)

    best_idx = np.nanargmax(psi)
    best_period = periods[best_idx]

    print(f"\n{label}")
    print("-" * len(label))
    print(f"N points = {len(time)}")
    print(f"Best hybrid period = {best_period:.6f} d")

    # Plot
    fig, ax = plt.subplots(3, 1, figsize=(8, 9), sharex=True)

    ax[0].plot(periods, ls_power, color="black", lw=1.2)
    ax[0].set_ylabel("LS power")
    ax[0].set_title(f"{label}: Lomb--Scargle")

    ax[1].plot(periods, lk_power, color="black", lw=1.2)
    ax[1].set_ylabel(r"$1/\Theta_{\rm LK}$")
    ax[1].set_title("Lafler--Kinman")

    ax[2].plot(periods, psi, color="black", lw=1.2)
    ax[2].axvline(best_period, color="red", ls="--", lw=1.3)
    ax[2].set_ylabel(r"Hybrid $\Psi$")
    ax[2].set_xlabel("Period [days]")
    ax[2].set_title("Hybrid statistic")

    for a in ax:
        a.set_xscale("log")
        a.minorticks_on()
        a.tick_params(which="both", direction="in", top=True, right=True)

    ax[2].text(
        0.05,
        0.9,
        rf"$P_{{\rm hybrid}} = {best_period:.5f}$ d",
        transform=ax[2].transAxes,
        fontsize=12,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8)
    )

    plt.tight_layout()
    plt.savefig(f"{label}_hybrid_periodogram.pdf", bbox_inches="tight")
    plt.savefig(f"{label}_hybrid_periodogram.png", dpi=600, bbox_inches="tight")
    plt.show()

    return best_period, periods, ls_power, lk_power, psi


In [ ]:
def plot_phase_folded(time, mag, period, label="dataset", nbins=20):
    """
    Phase-folded light curve with binned trend.
    """

    phase = ((time - np.nanmin(time)) / period) % 1.0

    bins = np.linspace(0, 1, nbins + 1)
    centres = 0.5 * (bins[:-1] + bins[1:])

    bin_mag = np.zeros(nbins)
    bin_err = np.zeros(nbins)

    for i in range(nbins):
        m = (phase >= bins[i]) & (phase < bins[i + 1])

        if np.sum(m) > 0:
            bin_mag[i] = np.nanmean(mag[m])
            bin_err[i] = np.nanstd(mag[m]) / np.sqrt(np.sum(m))
        else:
            bin_mag[i] = np.nan
            bin_err[i] = np.nan

    plt.figure(figsize=(8, 5))

    plt.scatter(
        phase,
        mag,
        s=12,
        alpha=0.25,
        color="gray",
        label="Data"
    )

    plt.errorbar(
        centres,
        bin_mag,
        yerr=bin_err,
        fmt="o-",
        color="black",
        lw=1.5,
        label="Binned"
    )

    plt.gca().invert_yaxis()
    plt.xlabel("Phase")
    plt.ylabel("Magnitude")
    plt.title(f"{label}: phase-folded light curve, P = {period:.5f} d")
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"{label}_phase_folded.pdf", bbox_inches="tight")
    plt.savefig(f"{label}_phase_folded.png", dpi=600, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# 2. ASAS-SN
# ============================================================

df_asas = pd.read_csv("AP58340417.csv")

df_asas = df_asas.dropna(subset=["hjd", "mag", "mag_err"])

df_asas = df_asas[
    (df_asas["mag_err"] > 0) &
    (df_asas["mag_err"] < 0.2)
].copy()

time_asas = df_asas["hjd"].values
mag_asas = df_asas["mag"].values
err_asas = df_asas["mag_err"].values

best_asas, p_asas, ls_asas, lk_asas, psi_asas = hybrid_period_search(
    time_asas,
    mag_asas,
    err_asas,
    label="ASAS-SN",
    min_period=0.2,
    max_period=100,
    n_periods=5000
)

plot_phase_folded(
    time_asas,
    mag_asas,
    best_asas,
    label="ASAS-SN"
)


In [ ]:
# ============================================================
# 3. ATLAS
# ============================================================

url = "https://fallingstar-data.com/forcedphot/static/results/job4162520.txt"

r = requests.get(url)

with open("atlas_lc.txt", "wb") as f:
    f.write(r.content)

cols = [
    "MJD", "m", "dm", "uJy", "duJy", "F", "err", "chi/N",
    "RA", "Dec", "x", "y", "maj", "min", "phi", "apfit",
    "mag5sig", "Sky", "Obs"
]

df_atlas = pd.read_csv(
    "atlas_lc.txt",
    sep=r"\s+",
    comment="#",
    names=cols,
    engine="python"
)

df_o = df_atlas[
    (df_atlas["F"] == "o") &
    np.isfinite(df_atlas["MJD"]) &
    np.isfinite(df_atlas["m"]) &
    np.isfinite(df_atlas["dm"]) &
    (df_atlas["dm"] > 0) &
    (df_atlas["dm"] < 0.2) &
    (df_atlas["m"] > 5) &
    (df_atlas["m"] < 25)
].copy()

time_atlas = df_o["MJD"].values
mag_atlas = df_o["m"].values
err_atlas = df_o["dm"].values

best_atlas, p_atlas, ls_atlas, lk_atlas, psi_atlas = hybrid_period_search(
    time_atlas,
    mag_atlas,
    err_atlas,
    label="ATLAS",
    min_period=0.2,
    max_period=100,
    n_periods=5000
)

plot_phase_folded(
    time_atlas,
    mag_atlas,
    best_atlas,
    label="ATLAS"
)

In [ ]:
# ============================================================
# 4. TESS
# ============================================================

target = "TIC 326446019"

lcc = lk.search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
).download_all()

lc = lcc.stitch().remove_nans().normalize()

lc_flat = lc.flatten(window_length=401).remove_nans()

time_f = np.asarray(lc_flat.time.value)
flux_f = np.asarray(lc_flat.flux.value)

finite = np.isfinite(time_f) & np.isfinite(flux_f)

time_f = time_f[finite]
flux_f = flux_f[finite]

# Remove strong positive flares before period search
med = np.nanmedian(flux_f)
mad = np.nanmedian(np.abs(flux_f - med))
sigma = 1.4826 * mad

flare_mask = flux_f > med + 5 * sigma
quiet_mask = ~flare_mask

time_tess = time_f[quiet_mask]
flux_tess = flux_f[quiet_mask]

# Convert normalized flux to relative magnitude
mag_tess = -2.5 * np.log10(flux_tess)

best_tess, p_tess, ls_tess, lk_tess, psi_tess = hybrid_period_search(
    time_tess,
    mag_tess,
    err=None,
    label="TESS",
    min_period=0.2,
    max_period=10,
    n_periods=5000
)

plot_phase_folded(
    time_tess,
    mag_tess,
    best_tess,
    label="TESS"
)


# ============================================================
# 5. Summary
# ============================================================

print("\nSummary of hybrid periods")
print("-------------------------")
print(f"ASAS-SN: {best_asas:.6f} d")
print(f"ATLAS:   {best_atlas:.6f} d")
print(f"TESS:    {best_tess:.6f} d")

In [ ]:
search_result = lk.search_lightcurve(
    target="TIC 326446019",
    mission="TESS",
    author="SPOC"
)

print(search_result)

In [ ]:
lcc = search_result.download_all()

for lc in lcc:
    sector = lc.meta["SECTOR"]

    print(f"Sector {sector}")

    lc = lc.remove_nans().normalize()

    freq, power = LombScargle(
        lc.time.value,
        lc.flux.value
    ).autopower(
        minimum_frequency=1/2.0,
        maximum_frequency=1/0.15,
        samples_per_peak=20
    )

    best_period = 1/freq[np.argmax(power)]

    print(f"Best period = {best_period:.6f} d")

In [ ]:
lc = search_result[0].download()
lc = search_result[1].download()
lc = search_result[3].download()